<a href="https://colab.research.google.com/github/MichaelZimm20/xn-sitstayforever-semantic-attention/blob/main/notebooks/xn_sitstayforever_models_and_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ------ IMPORTS & SETUP ------
'''
Grad-CAM implementation adapted from:
- Selvaraju et al. (2017), "Grad-CAM: Visual Explanations from Deep Networks
  via Gradient-based Localization" (original Grad-CAM formulation)
- Michael's own prior implementation, AAI6640 Applied Deep Learning final
  project (Brain Tumor MRI CNN classification), Spring 2026


This section adapts the GradCAM concepts to run against the CLIP's visual encoder rather than a standard
image classifer.
- uses cosine similarity to target keyword as the backward pass signal instead of classification logit
- implementing a wrapper using the cosine similarity score to compute the CLIP image/text embedding back to the GradCAM hooks

Purpose:
IS to generate an attention heatmap showing where the models attentions lies when evalauting an image against key words. Theese
keywords are common search terms and listing words that people use for their products
- Example " groomer approved dry shampoo"
This wll help connect the keyword-alignment tool in Notebook 1 to GradCAM
'''


# IMPORTS
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# IMPORT CLIP
# Needed for RN50, it has conv layers that GradCAM utilizes
# Installs for missing dependencies not natively supported
!pip install open-clip-torch -q

print("Open CLIP install successful!")
print('~'*50)
# check cpu versus gpu on device using torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


# DRIVE SETUP AND STRUCTURE
BASE_DIR = '/content/drive/MyDrive/xn-sitstayforever'
DATASET_DIR = f'{BASE_DIR}/datasets'
OUTPUTS_DIR = f'{BASE_DIR}/outputs'
CHECKPOINTS_DIR = f'{BASE_DIR}/checkpoints'

# lOAD NOTEBOOK 1 CSVs outputs
  # training set
clip_train = pd.read_csv(f'{OUTPUTS_DIR}/clip_train.csv')
  # validation set
clip_ssf_eval = pd.read_csv(f'{OUTPUTS_DIR}/clip_ssf_eval.csv')
  # merge train and eval clip scores
clip_scores_merged = pd.concat([clip_train, clip_ssf_eval], ignore_index=True)

# reload notebook 1, 20 keywords used for CLIP and keyword alignment
text_queries = clip_train['all_target_keywords'].iloc[0].split(',')


print('=' * 50)
print('Imports and Setup are successful!')
print('=' * 50)
print(f'Loaded {len(clip_scores_merged)} CLIP images total')



Open CLIP install successful!
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Using device: cpu
Imports and Setup are successful!
Loaded 54 CLIP images total


In [6]:
'''Smoke Test for verfiying directories, paths, and datasets are loading properly'''
import os

# File path smoke test, since files are in a google drive and mounted
# for item in os.listdir(BASE_DIR):
#   print(repr(item))
print('=' * 50)
for item in os.listdir(DATASET_DIR):
  print(item)
print('=' * 50)
# check exact folder name including hidden characters
items = os.listdir(BASE_DIR)
for item in items:
  print(repr(item))

# check it path exists
print(os.path.exists(DATASET_DIR))

SSF_CV_Dataset.xlsx
pet_cv_dataset_full.xlsx
product_images
'datasets'
'checkpoints'
'outputs'
True
